In [15]:
import datetime as dt

import optuna
import pandas as pd
from moexalgo import CandlePeriod

import luigi
from src.auction_model import RecalibrationMode, AuctionModelInference
from config import LUIGI_OUTPUT_DIR as BASE_DIR
from src.qr import get_mos_dates
from src.tasks import YieldCandles, BondType, Board, BondsDescription
from src.instruments import OFZ

luigi.interface.core.log_level = "WARNING"
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [28]:
from src.qr import get_mos_dates_range
from src.tasks import Candles
from src.graph import InMemoryTask, DailyMixin


class ADV(InMemoryTask, DailyMixin):
    nb_days = luigi.IntParameter(default=10)

    def requires(self):
        return [self.clone(Candles, date=date, board=Board.TQOB, period=CandlePeriod.ONE_HOUR) for date in get_mos_dates_range(self.prev_date, self.nb_days)]

    def produce_output(self):
        value_df = pd.concat(t.read_output()['volume'].unstack().T.between_time(dt.time(10), dt.time(19)) for t in self.requires())
        return value_df.groupby(value_df.index.to_series().dt.date).sum().mean().to_frame('adv') * 1e3

In [29]:
dates = get_mos_dates(dt.date(2024, 10, 1), dt.date(2026, 2, 19), incl_end=False)
dates[0], dates[-1]

(datetime.date(2024, 10, 1), datetime.date(2026, 2, 18))

In [18]:
luigi.build([AuctionModelInference(date=date, out_dir=LUIGI_OUTPUT_DIR, time_of_day=dt.timedelta(hours=18), recalibration_mode=RecalibrationMode.QUARTERLY) for date in dates], local_scheduler=True)
inference_df = pd.concat(AuctionModelInference(date=date, out_dir=LUIGI_OUTPUT_DIR, time_of_day=dt.timedelta(hours=18), recalibration_mode=RecalibrationMode.QUARTERLY).read_output() for date in dates)

inference_df['proba_norm'] = inference_df['proba'] / inference_df.groupby('prediction_ts')['proba'].transform('sum') * 2
inference_df['begin'] = inference_df['prediction_ts']

In [37]:
nb_days=1
luigi.build([ADV(date=date, out_dir=LUIGI_OUTPUT_DIR, nb_days=nb_days) for date in dates], local_scheduler=True)
adv_df = pd.concat(ADV(date=date, out_dir=LUIGI_OUTPUT_DIR, nb_days=nb_days).read_output().assign(begin=pd.Timestamp(date) + pd.Timedelta(hours=18)) for date in dates)
adv_df = adv_df.reset_index().set_index(['secid', 'begin'])

In [38]:
yields_df = pd.concat(YieldCandles(out_dir=LUIGI_OUTPUT_DIR, date=d, period=CandlePeriod.ONE_HOUR, bond_type=BondType.FIX, board=Board.TQOB).read_output() for d in dates)

In [39]:
df = yields_df.query('end.dt.hour == 18').join(inference_df.set_index(['secid', 'begin'])[['proba', 'proba_norm', 'remaining_size_mln']]).join(adv_df)

In [40]:
from src.tasks import BondMetricsAtVWAP

In [59]:
yields_df.sort_values('volume', ascending=False)

open      close       high        low  \
secid        begin                                                             
SU26246RMFS7 2025-03-05 11:00:00  16.310150  16.310150  16.276182  16.318473   
SU26229RMFS3 2025-10-24 14:00:00  18.454135  18.454135  18.427851  18.822744   
SU26248RMFS3 2025-09-12 13:00:00  14.038833  14.193931  14.025758  14.215175   
             2025-06-19 10:00:00  15.344712  15.312016  15.307706  15.368013   
             2024-12-20 14:00:00  16.990268  16.655477  16.463566  16.992669   
...                                     ...        ...        ...        ...   
             2024-10-15 09:00:00  16.936475  16.936475  16.936475  16.936475   
SU26226RMFS9 2024-10-30 09:00:00  20.596156  20.596156  20.596156  20.596156   
SU26233RMFS5 2025-08-28 08:00:00  13.864558  13.864558  13.864558  13.864558   
SU26230RMFS1 2025-11-06 06:00:00  14.736458  14.736458  14.736458  14.736458   
SU26244RMFS2 2025-08-20 08:00:00  13.538546  13.538546  13.538546  13.538546   

                                         value      volume  \
secid        begin                                           
SU26246RMFS7 2025-03-05 11:00:00  1.644710e+10  20329072.0   
SU26229RMFS3 2025-10-24 14:00:00  1.463891e+10  14707449.0   
SU26248RMFS3 2025-09-12 13:00:00  1.059455e+10  11662417.0   
             2025-06-19 10:00:00  8.458682e+09   9963397.0   
             2024-12-20 14:00:00  6.329717e+09   8054756.0   
...                                        ...         ...   
             2024-10-15 09:00:00  7.737200e+02         1.0   
SU26226RMFS9 2024-10-30 09:00:00  8.189900e+02         1.0   
SU26233RMFS5 2025-08-28 08:00:00  6.056400e+02         1.0   
SU26230RMFS1 2025-11-06 06:00:00  6.140000e+02         1.0   
SU26244RMFS2 2025-08-20 08:00:00  9.057800e+02         1.0   

                                                 end  weighted_avg  
secid        begin                                                  
SU26246RMFS7 2025-03-05 11:00:00 2025-03-05 11:59:59     16.309088  
SU26229RMFS3 2025-10-24 14:00:00 2025-10-24 14:59:59     18.796365  
SU26248RMFS3 2025-09-12 13:00:00 2025-09-12 13:59:59     14.176684  
             2025-06-19 10:00:00 2025-06-19 10:59:59     15.328748  
             2024-12-20 14:00:00 2024-12-20 14:59:59     16.674886  
...                                              ...           ...  
             2024-10-15 09:00:00 2024-10-15 09:59:59     16.936475  
SU26226RMFS9 2024-10-30 09:00:00 2024-10-30 09:59:59     20.596156  
SU26233RMFS5 2025-08-28 08:00:00 2025-08-28 08:59:59     13.864558  
SU26230RMFS1 2025-11-06 06:00:00 2025-11-06 06:59:59     14.736458  
SU26244RMFS2 2025-08-20 08:00:00 2025-08-20 08:59:59     13.538546  

[159283 rows x 8 columns]

In [57]:
yields_df.assign(dt=yields_df.end.dt.date).reset_index().groupby(['dt', 'secid'][::])['volume'].sum().loc[dt.date(2025, 3, 5)].sort_values(ascending=False)

secid
SU26246RMFS7    53528016.0
SU26248RMFS3     9728087.0
SU26243RMFS4     4766500.0
SU26230RMFS1     3571905.0
SU26247RMFS5     2886841.0
SU26238RMFS4     2677210.0
SU26244RMFS2     1543494.0
SU26245RMFS9     1529886.0
SU26225RMFS1     1290137.0
SU26233RMFS5     1058655.0
SU26221RMFS0     1049650.0
SU26242RMFS6      940198.0
SU26239RMFS2      668331.0
SU26240RMFS0      490383.0
SU26237RMFS6      426043.0
SU26212RMFS9      419583.0
SU26232RMFS7      368470.0
SU26234RMFS3      320536.0
SU26228RMFS5      278255.0
SU26207RMFS9      275136.0
SU26236RMFS8      246784.0
SU26226RMFS9      192533.0
SU26219RMFS4       99142.0
SU26229RMFS3       98461.0
SU26224RMFS4       94417.0
SU26235RMFS0       70833.0
SU26241RMFS8       66316.0
SU26218RMFS6       60846.0
Name: volume, dtype: float64

In [60]:
secids = [OFZ[26245].secid, OFZ[26246].secid, OFZ[26225].secid][1:]
secids = [OFZ[26238].secid, OFZ[26230].secid]
df['adv'].loc[secids].unstack().T.fillna(0.).plot(backend='plotly')

In [61]:
(df['weighted_avg'].loc[secids].unstack().T.diff(axis=1)).plot(backend='plotly')

In [25]:
(df['remaining_size_mln'].loc[secids].unstack().T.fillna(0.)).plot(backend='plotly')

In [26]:
from src.instruments import symbol_to_series
metrics = BondMetricsAtVWAP(date=dt.date.today(), out_dir=LUIGI_OUTPUT_DIR, bond_type=BondType.FIX).read_output()
metrics['series'] = symbol_to_series(metrics.index.to_series())
metrics = metrics.set_index('series')
from src.tasks import PlacedSizeHistory
old_df = PlacedSizeHistory(out_dir=LUIGI_OUTPUT_DIR).read_output().query('timestamp < 20220301 and type == "ОФЗ-ПД"').groupby('series').last()['placed_size_mln'].to_frame('old_size_mln')
tod_df = PlacedSizeHistory(out_dir=LUIGI_OUTPUT_DIR).read_output().query('timestamp < 20320301 and type == "ОФЗ-ПД"').groupby('series').last()['placed_size_mln'].to_frame('cur_size_mln')
tod_df.join(old_df).join(metrics.dropna()[['duration', 'price', 'yield']], how='right').sort_values('duration').pipe(lambda x: x.assign(pct_new_size=(x['cur_size_mln'] - x['old_size_mln'].fillna(0)) / x['cur_size_mln']))

,cur_size_mln,old_size_mln,duration,price,yield,pct_new_size
series,,,,,,
26219,364091.149,350000.149,0.520633,96.805355,14.577600,0.038702
26226,368567.898,349999.898,0.577721,96.639107,14.459381,0.050379
26207,371813.713,350000.713,0.903336,95.488210,13.946438,0.058666
26232,450000.000,450000.000,1.502350,89.494904,13.992194,0.000000
26212,350000.000,350000.000,1.776057,89.484324,14.012833,0.000000
26236,500000.000,500000.000,2.059304,85.147808,14.179056,0.000000
26237,419119.019,351097.019,2.669505,83.124228,14.149206,0.162298
26224,450000.000,350000.000,2.852703,82.691347,14.180749,0.222222
26242,529356.615,NaN,2.890806,87.213730,14.202845,1.000000


In [232]:
inference_tasks = [AuctionModelInference(out_dir=LUIGI_OUTPUT_DIR, date=d, time_of_day=dt.timedelta(hours=18, minutes=50),
                                         recalibration_mode=RecalibrationMode.QUARTERLY) for d in dates]
luigi.build(inference_tasks, local_scheduler=True)
inference_df = pd.concat([t.read_output() for t in inference_tasks])

In [233]:
luigi.build([AuctionModelInference(out_dir=LUIGI_OUTPUT_DIR, date=d, time_of_day=dt.timedelta(hours=18, minutes=50), recalibration_mode=RecalibrationMode.QUARTERLY) for d in dates[-1:]], local_scheduler=True)

True

In [371]:
from sklearn.linear_model import LinearRegression
import numpy as np
from dataclasses import dataclass


@dataclass
class InterpolationParams:
    bandwidth: float = 2.
    value_bln_cap: float = 0.5
    max_extrapolation: float = 0.5


def calc_interpolated_ytm(df, target_duration, params):
    """
    Compute YTM over a vector of durations using liquidity-weighted local linear regression.
    """

    duration_diff = df["duration"] - target_duration
    w_dist = (1.0 - duration_diff.abs() / params.bandwidth).clip(lower=0.)
    w_liq = (df['value_bln_m20'] / params.value_bln_cap).clip(lower=1.0, upper=1.0)
    w = w_dist * w_liq

    if w[duration_diff >= -params.max_extrapolation].sum() == 0 or w[duration_diff <= params.max_extrapolation].sum() == 0:
        return np.nan

    X = df[["duration"]]
    model = LinearRegression()
    model.fit(X, df["vway"], sample_weight=w)

    return model.predict(pd.DataFrame({"duration": [target_duration]}))[0]

In [ ]:
df

In [ ]:
BondMetricsAtVWAP

In [381]:
task = BondsDescription(date=dt.date(2026, 2, 27), out_dir=LUIGI_OUTPUT_DIR)
task.clone(BondMetricsAtVWAP, bond_type=BondType.FIX)
df = task.read_output()